# 08 · Do the bottlenecks keep the pose information, and save calibration data?

**Investigation ladder** (see [`EVALUATION.md`](../EVALUATION.md)):
- **Stage 6 · Bottlenecks and data efficiency: do the bottlenecks keep information and reduce data needs?**

**Read first:** the THANOS proposal (Aim 2, the 2 × 2 design); Semedo et al. 2019, *Cortical areas interact through a communication subspace* (Neuron) for the idea that the part of one signal that predicts another is not its biggest part; notebooks [05](../05_link_to_movement/05_link_to_movement.ipynb) and [06](../06_shared_subspace/06_shared_subspace.ipynb).

**We expect:** a bottleneck chosen **for the pose** (the shared EMG–pose subspace from notebook 06) keeps nearly all of the decoder's information at a small size, while variance-based NMF synergies of the same size do no better than a random basis. **Wrong if:** the pose-informed bottleneck is no better than random, or every bottleneck costs several degrees.

**Run this after stages 4 and 4b** (notebooks 05 and 06). It reuses notebook 05's metrics, ridge and lag features ([`ref05.py`](../05_link_to_movement/ref05.py)), notebook 02's NMF and PCA ([`ref02.py`](../02_how_many_synergies/ref02.py)) and notebook 06's reduced-rank regression ([`ref06.py`](../06_shared_subspace/ref06.py)). Paste your own versions into the setup cell if you like.

**How this notebook works** (the same in every notebook)
- **`# TODO`** cells are yours. Each has a docstring with the input and output shapes. You write the core
  algorithms yourself in NumPy.
- Under every TODO there are three **💡 hints** that open one at a time: the plan in words, the tools with a tiny
  example, then a skeleton with blanks (`____`).
- After each TODO a **✅ check cell** compares your answer with the reference in [`ref08.py`](ref08.py), then a **📊 plot**
  shows what your code did. Other notebooks use the reference, so your blanks never break theirs.
- **✍️ questions** are for your own words, in the markdown cell under them.
- The **Result** cell at the end is the **gate**: it decides whether the next stage of the investigation is worth running.

## 🎯 Goal and milestones
*Draft milestones: agree them as a team on Friday, then edit here.*

**Proposal:** 🧪 pilot of **Aim 2** (the 2 × 2 bottleneck design and calibration curves), on one person, with the linear lagged decoder of notebook 05.

**Goal:** find out whether squeezing the EMG input and the pose output through a few dimensions keeps the information the decoder uses, whether it matters **how** those dimensions are chosen (by variance or for the pose), and whether a bottlenecked decoder needs less data on a new task.

**Why:** the proposal's hypothesis (H1) is that bottlenecks cut calibration on new users. One person can't test that, but it can show whether the bottlenecks keep the information at all, which must hold before stage 7 is worth running.

**Milestones**
1. The 2 × 2 with variance-based bottlenecks (NMF synergies, eigenpostures) and random controls
2. The same with pose-informed bottlenecks (reduced-rank regression, notebook 06)
3. Bottleneck size sweep: k = 2, 3, 4, 8 for each kind
4. Calibration curves on held-out stages, with everything refit per fold

**Gate (decides the next stage):** stage 6 passes if a bottleneck of a few dimensions stays within ~0.5° of no bottleneck, clearly beats a random bottleneck of the same size, and its calibration curve is at or below the no-bottleneck curve.

**Done when:** the stage 6 gate table filled in, with a pass / fail and a reason.

In [ ]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    else:
        !git -C {ROOT} pull -q                 # pick up files pushed since the first clone
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

_d = ROOT / "notebooks" / "experiments" / "08_bottlenecks"
assert (_d / "ref08.py").exists(), f"{_d}/ref08.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref08", None)
_d = ROOT / "notebooks" / "experiments" / "05_link_to_movement"
assert (_d / "ref05.py").exists(), f"{_d}/ref05.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref05", None)
_d = ROOT / "notebooks" / "experiments" / "06_shared_subspace"
assert (_d / "ref06.py").exists(), f"{_d}/ref06.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref06", None)
_d = ROOT / "notebooks" / "experiments" / "02_how_many_synergies"
assert (_d / "ref02.py").exists(), f"{_d}/ref02.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref02", None)
_d = ROOT / "notebooks" / "experiments" / "common"
sys.path.insert(0, str(_d)); sys.modules.pop("protocol", None)
import protocol                 # shared evaluation protocol: sessions, folds, baselines, nulls, CIs
import ref08 as ref          # the reference answers, used only by the ✅ check cells

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01's prep.py (data/ is never committed)

# Tools from earlier notebooks (paste your own versions here instead, if you like)
from ref05 import (angular_error, r2, score, ridge_fit, ridge_predict, fit_and_score_ridge, choose_alpha,
                   lag_features, per_recording_median)        # notebook 05: metrics, ridge, lags
from ref02 import nmf_fit, nmf_transform, pca_fit, pca_transform, pca_inverse   # notebook 02: NMF, PCA
import ref06                                                   # notebook 06: reduced-rank regression

In [ ]:
# ── Load the shared data (rebuilds it with prep.py if missing or built by an older version) ──
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(ROOT / "notebooks" / "experiments" / "01_data_prep")); sys.modules.pop("prep", None)
import prep
if not prep.is_current(PROCESSED):
    subprocess.run([sys.executable, "notebooks/experiments/01_data_prep/prep.py"], check=True)
d = np.load(PROCESSED)

emg        = d["emg"].astype(np.float64)   # (N, 16)  EMG envelope, 50 samples/s, non-negative
pose       = d["pose"].astype(np.float64)  # (N, 20)  joint angles in degrees (low-passed at 6 Hz)
pose_valid = d["pose_valid"]               # (N,)     False where pose tracking failed
rec        = d["rec"]                      # (N,)     which recording (0–29) each row comes from
split      = d["split"]                    # (N,)     0 = train, 1 = test, -1 = gap
stage_of_rec, filename = d["stage"], d["filename"]   # (30,) per recording

train, test = split == 0, split == 1
tr, te = train & pose_valid, test & pose_valid
SEED = 0

# The stage-4 decoder from notebook 05 (given): ridge on log-EMG with 40, 100, 200 and 300 ms of history
log_emg = np.log(emg + 1e-3)                                # (N, 16)
LAGS = (2, 5, 10, 15)
F_lag = lag_features(log_emg, rec, LAGS)                    # (N, 80)
alpha_lag = choose_alpha(F_lag[tr], pose[tr], rec[tr])
results = {"global median": score(pose[te], np.tile(np.median(pose[tr], axis=0), (te.sum(), 1))),
           "per-recording median": score(pose[te], per_recording_median(pose, rec, tr)[te]),
           "16 ch + lags (log)": fit_and_score_ridge(F_lag[tr], pose[tr], F_lag[te], pose[te], alpha_lag)}
X_rrr = ref06.lag_features(log_emg, rec, [10])              # (N, 16): log-EMG 200 ms before each frame (notebook 06)
for name, s in results.items():
    print(f"{name:22s} {s['angle_deg']:6.2f}°   R² = {s['r2']:6.3f}")

### The variables you start with
| Name | Shape | What it is |
|---|---|---|
| `emg`, `log_emg` | (N, 16) | EMG envelope, and its log |
| `pose` | (N, 20) | joint angles in degrees |
| `tr`, `te` | (N,) bool | time split (first 80% / last 20% of each recording), valid pose only |
| `F_lag`, `alpha_lag` | (N, 80), float | the stage-4 decoder's features (log-EMG now and 40–300 ms ago) and its ridge penalty |
| `X_rrr` | (N, 16) | log-EMG 200 ms before each frame: what notebook 06 used to find the shared subspace |
| `results` | dict | the baselines and the no-bottleneck decoder ("neither" below) |

Letters in the docstrings: **k** = size of the sEMG bottleneck, **K_POSE** = size of the pose bottleneck.

## Before you run anything: your prediction

Write what you expect and why (2–3 sentences). Compare at the end. Surprises are the interesting part.

> *Prediction:*

---
## Why this is the second attempt at the 2 × 2
The first version used ridge from the EMG **at one instant**. That decoder was no better than guessing the median
pose, so every condition, with or without bottlenecks, real or random, predicted roughly the median pose and scored
the same. A bottleneck can only *lose* information the decoder was using; when the decoder uses none, the comparison
is empty. Here the decoder is notebook 05's **lagged log-EMG** decoder, which beats the baselines and a shifted null.

| | Pose out: all 20 angles | Pose out: K_POSE directions |
|---|---|---|
| **sEMG in: all 16 channels** | neither | pose only |
| **sEMG in: k directions** | sEMG only | **both** |

**Two ways to choose the k (or K_POSE) directions:**
- **By variance** (the proposal's choice): NMF **synergies** keep the EMG patterns that rebuild the EMG best;
  PCA **eigenpostures** keep the pose patterns that rebuild the pose best. Neither looks at the other signal.
- **For prediction** (notebook 06): **reduced-rank regression (RRR)** keeps the k EMG directions, and the K_POSE
  pose directions, through which the EMG predicts the pose best: the *shared* or *communication* subspace.

Notebook 06 found the shared subspace is small and lies **no closer to the synergies than a random subspace**. If
so, the synergy bottleneck throws away exactly the part of the EMG the decoder needs, and the RRR bottleneck should
not. **Controls of the same size:** a random non-negative EMG basis (like NMF), a random orthonormal EMG basis
(like RRR), and random orthonormal pose directions.

---
## Task 1 · The 2 × 2 with synergies and eigenpostures
**sEMG bottleneck:** fit NMF on the **training** EMG (`emg[train]`: all training frames, including tracking
dropouts, since NMF never looks at the pose). To keep it fast, fit on every 5th frame with 200 rounds. Compute the
activations `C` for **all** N frames with `W` fixed, then build the features exactly like the stage-4 decoder:
`lag_features(np.log(C + 1e-3), rec, LAGS)`.

**Pose bottleneck:** ridge predicts the K_POSE **scores** of the pose (PCA fit on `pose[tr]` only), and
`pca_inverse` turns predicted scores back into 20 angles. Notebook 02 found the pose is **not** very
low-dimensional here, so the pose bottleneck has a **ceiling**: even a perfect decoder can't beat the error of the
true poses squeezed through K_POSE directions. You'll compute that ceiling in Task 2.

In [ ]:
K = 4          # sEMG bottleneck size (notebook 02 chose 4–5 from VAF)
K_POSE = 6     # pose bottleneck size (replace with yours from notebook 02)
rng = np.random.default_rng(SEED)

_, W_syn, _ = nmf_fit(emg[train][::5], K, n_iter=200, seed=SEED)     # (K, 16) synergies
C_syn = nmf_transform(emg, W_syn, n_iter=200, seed=SEED)              # (N, K) activations of every frame
W_rand = rng.random((K, 16)); W_rand /= np.linalg.norm(W_rand, axis=1)[:, None]   # random non-negative basis

pose_mean, pose_comps = pca_fit(pose[tr], K_POSE)                     # (20,), (K_POSE, 20) eigenpostures
rand_comps = np.linalg.qr(rng.normal(size=(20, K_POSE)))[0].T         # (K_POSE, 20) random orthonormal directions


def fit_and_score_pose_bottleneck(F_train: np.ndarray, Y_train: np.ndarray, F_test: np.ndarray,
                                  Y_test: np.ndarray, pose_mean: np.ndarray, pose_comps: np.ndarray,
                                  alpha: float = 1.0) -> dict[str, float]:
    """Decoder with a pose bottleneck: features → K_POSE scores → 20 angles.

    Parameters
    ----------
    F_train, Y_train : (T_train, d), (T_train, 20) on the training frames
    F_test, Y_test   : (T_test, d), (T_test, 20) on the test frames
    pose_mean, pose_comps : (20,), (K_POSE, 20) with orthonormal rows (eigenpostures, RRR or random directions)
    alpha            : ridge penalty

    Returns
    -------
    score(Y_test, prediction). Steps: targets = pca_transform(Y_train, …) → ridge_fit(F_train, targets)
    → ridge_predict on F_test → pca_inverse(…) → score against Y_test.
    """
    # TODO
    raise NotImplementedError


F_syn = None        # (N, K * 5): TODO lag_features(np.log(C_syn + 1e-3), rec, LAGS)
F_rand = None       # (N, K * 5): TODO the same with emg @ W_rand.T in place of C_syn
alpha_syn = None    # TODO: choose_alpha(F_syn[tr], pose[tr], rec[tr])
alpha_rand = None   # TODO: choose_alpha(F_rand[tr], pose[tr], rec[tr])
two_by_two: dict[str, dict[str, float]] = {}
# TODO: "neither" = results["16 ch + lags (log)"]; "sEMG only" = ridge on F_syn;
#       "pose only" = pose bottleneck on F_lag (alpha_lag); "both" = pose bottleneck on F_syn (alpha_syn);
#       "random sEMG" = ridge on F_rand; "random both" = pose bottleneck on F_rand with rand_comps

#### 💡 Stuck on Task 1? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `fit_and_score_pose_bottleneck`: training poses → K_POSE scores (the ridge targets) → fit → predict test
   scores → back to 20 angles → score.
2. Build `F_syn` and `F_rand` exactly like `F_lag` (log, then lags) and choose their alphas on `tr`.
3. Six entries in `two_by_two`; "neither" is a copy of a stage-4 result.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- Ridge works for any number of target columns: here J = K_POSE.
- `pca_transform(Y, mean, comps)` → (T, K_POSE); `pca_inverse(S, mean, comps)` → (T, 20). Any orthonormal
  `comps` works, including random ones.

```python
S = np.array([[1.0, 2.0]]); comps = np.eye(2, 3); mean = np.zeros(3)
S @ comps + mean          # [[1., 2., 0.]]: what pca_inverse does
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def fit_and_score_pose_bottleneck(F_train, Y_train, F_test, Y_test, pose_mean, pose_comps, alpha=1.0):
    targets = pca_transform(____, pose_mean, pose_comps)
    w, b = ridge_fit(F_train, ____, alpha)
    Y_hat = pca_inverse(ridge_predict(____, w, b), pose_mean, pose_comps)
    return score(Y_test, ____)

F_syn = lag_features(np.log(C_syn + 1e-3), rec, LAGS)
F_rand = lag_features(np.log(emg @ W_rand.T + 1e-3), rec, ____)
alpha_syn = choose_alpha(F_syn[tr], pose[tr], rec[tr])
alpha_rand = choose_alpha(F_rand[tr], pose[tr], rec[tr])
two_by_two["neither"] = results["16 ch + lags (log)"]
two_by_two["sEMG only"] = fit_and_score_ridge(F_syn[tr], pose[tr], F_syn[te], pose[te], alpha_syn)
two_by_two["pose only"] = fit_and_score_pose_bottleneck(F_lag[tr], pose[tr], F_lag[te], pose[te], pose_mean, pose_comps, alpha_lag)
two_by_two["both"] = fit_and_score_pose_bottleneck(____[tr], pose[tr], ____[te], pose[te], pose_mean, pose_comps, alpha_syn)
two_by_two["random sEMG"] = fit_and_score_ridge(F_rand[tr], pose[tr], F_rand[te], pose[te], alpha_rand)
two_by_two["random both"] = fit_and_score_pose_bottleneck(F_rand[tr], pose[tr], F_rand[te], pose[te], pose_mean, ____, alpha_rand)
```

</details>

In [ ]:
# ✅ Check Task 1
assert F_syn.shape == (len(emg), K * (1 + len(LAGS))) and F_rand.shape == F_syn.shape
for name, F_a, a, comps in [("pose only", F_lag, alpha_lag, pose_comps), ("both", F_syn, alpha_syn, pose_comps),
                            ("random both", F_rand, alpha_rand, rand_comps)]:
    expect = ref.fit_and_score_pose_bottleneck(F_a[tr], pose[tr], F_a[te], pose[te], pose_mean, comps, a)
    assert np.isclose(two_by_two[name]["angle_deg"], expect["angle_deg"]), \
        f"{name}: {two_by_two[name]['angle_deg']:.3f}°, reference {expect['angle_deg']:.3f}°"
assert set(two_by_two) == {"neither", "sEMG only", "pose only", "both", "random sEMG", "random both"}
for name, s in {**results, **two_by_two}.items():
    print(f"{name:22s} {s['angle_deg']:6.2f}°   R² = {s['r2']:6.3f}")
print("✅ 2 × 2 with synergies and eigenpostures done")

✍️ **Q1.** Which bottleneck costs more, the synergy one (k = 4) or the eigenposture one (K_POSE = 6)? Does the synergy
bottleneck beat the random non-negative basis of the same size? If not, what does that mean for the claim that
synergies are a *useful* decoder input?

> *Your answer:*

---
## Task 2 · Bottlenecks chosen for the pose: the shared subspace
Notebook 06's reduced-rank regression (RRR) fits the best map from EMG to pose that passes through only `rank`
dimensions. `ref06.rrr_fit(X, Y, rank)` returns `B_r, x_mean, y_mean, V`, and its two sides give two bottlenecks:

- **EMG side:** `ref06.predictive_subspace(B_r, k)` → `Q` (16 × k), the k channel directions the map actually
  uses. The bottlenecked EMG is `(log_emg - x_mean) @ Q`: k latent signals, each a weighted sum of the channels
  (weights can be negative, unlike NMF). Lag them like any other features.
- **Pose side:** `V` (20 × K_POSE), the pose directions the EMG predicts best. `V.T` has the same layout as
  eigenpostures, so it plugs straight into `fit_and_score_pose_bottleneck` (with `y_mean` as the mean).

Fit RRR on `X_rrr[tr]` (log-EMG 200 ms earlier) and `pose[tr]`, training frames only.

**The pose bottleneck's ceiling.** Squeeze the **true** test poses through the K_POSE directions and back. Their
error is the best *any* decoder with that bottleneck could reach. Eigenpostures are built to make this ceiling as
low as possible; RRR directions aren't, but they keep the part of the pose the EMG can actually predict.

**Be fair to NMF:** RRR **sees the pose** when it chooses its directions; NMF never does. So this is not "which
synergies are more real", but "which way of choosing a few dimensions keeps the decoder's information".

In [ ]:
def rrr_channel_basis(X: np.ndarray, Y: np.ndarray, k: int, alpha: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    """EMG-side bottleneck chosen for the pose.

    Parameters
    ----------
    X : (T, 16) float, lagged log-EMG on the fit frames (e.g. X_rrr[tr])
    Y : (T, 20) float, pose on the same frames
    k : int, bottleneck size

    Returns
    -------
    Q      : (16, k) float, orthonormal columns: ref06.predictive_subspace of a rank-k ref06.rrr_fit
    x_mean : (16,) float, the mean of X (rrr_fit returns it)
    """
    # TODO
    raise NotImplementedError


def rrr_pose_basis(X: np.ndarray, Y: np.ndarray, K: int, alpha: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    """Pose-side bottleneck chosen for the EMG: (y_mean (20,), comps (K, 20)) = (y_mean, V.T) of a rank-K rrr_fit."""
    # TODO
    raise NotImplementedError


def projection_ceiling(Y: np.ndarray, pose_mean: np.ndarray, pose_comps: np.ndarray) -> float:
    """Angular error of the TRUE poses Y squeezed through the pose bottleneck: pca_inverse(pca_transform(Y, …), …)."""
    # TODO
    raise NotImplementedError


Q_rrr, x_mean_rrr = None, None     # TODO: rrr_channel_basis(X_rrr[tr], pose[tr], K)
F_rrr = None                       # (N, K * 5): TODO lag_features((log_emg - x_mean_rrr) @ Q_rrr, rec, LAGS)
rrr_mean, rrr_comps = None, None   # TODO: rrr_pose_basis(X_rrr[tr], pose[tr], K_POSE)
Q_orth = ref06.random_subspace(16, K, np.random.default_rng(K))                   # random orthonormal EMG basis (given)
F_orth = lag_features((log_emg - log_emg[train].mean(axis=0)) @ Q_orth, rec, LAGS)  # given
alpha_rrr, alpha_orth = None, None # TODO: choose_alpha for F_rrr and F_orth on tr
# TODO: two_by_two["sEMG only (RRR)"]  = ridge on F_rrr
#       two_by_two["pose only (RRR)"]  = pose bottleneck on F_lag with (rrr_mean, rrr_comps), alpha_lag
#       two_by_two["both (RRR)"]       = pose bottleneck on F_rrr with (rrr_mean, rrr_comps), alpha_rrr
#       two_by_two["random orth sEMG"] = ridge on F_orth
ceilings: dict[str, float] = {}    # TODO: "eigenpostures", "RRR pose", "random" → projection_ceiling(pose[te], …)

#### 💡 Stuck on Task 2? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `rrr_channel_basis`: call `ref06.rrr_fit(X, Y, k, alpha)`; keep `B_r` and `x_mean`; return
   `ref06.predictive_subspace(B_r, k)` and `x_mean`.
2. `rrr_pose_basis`: call `rrr_fit` with rank K; return `y_mean` and `V.T`.
3. `projection_ceiling`: centre, project onto the rows of `comps`, project back, add the mean, angular error.
4. Then the same four steps as Task 1 for the new feature sets and pose basis.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- Unpacking and ignoring: `B_r, x_mean, _, _ = ref06.rrr_fit(X, Y, k)`.
- Projecting onto orthonormal rows `C` (K × 20) and back: `(Y - m) @ C.T @ C + m`.

```python
C = np.array([[1.0, 0.0]]); Y = np.array([[3.0, 4.0]]); m = np.zeros(2)
(Y - m) @ C.T @ C + m      # [[3., 0.]]: only the part along the first axis survives
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def rrr_channel_basis(X, Y, k, alpha=1.0):
    B_r, x_mean, _, _ = ref06.rrr_fit(X, Y, ____, alpha)
    return ref06.predictive_subspace(B_r, ____), x_mean

def rrr_pose_basis(X, Y, K, alpha=1.0):
    _, _, y_mean, V = ref06.rrr_fit(X, Y, K, alpha)
    return y_mean, ____

def projection_ceiling(Y, pose_mean, pose_comps):
    return angular_error(Y, pca_inverse(pca_transform(Y, pose_mean, pose_comps), ____, ____))

Q_rrr, x_mean_rrr = rrr_channel_basis(X_rrr[tr], pose[tr], K)
F_rrr = lag_features((log_emg - x_mean_rrr) @ ____, rec, LAGS)
rrr_mean, rrr_comps = rrr_pose_basis(X_rrr[tr], pose[tr], ____)
alpha_rrr = choose_alpha(F_rrr[tr], pose[tr], rec[tr]); alpha_orth = choose_alpha(F_orth[tr], pose[tr], rec[tr])
two_by_two["sEMG only (RRR)"] = fit_and_score_ridge(F_rrr[tr], pose[tr], F_rrr[te], pose[te], alpha_rrr)
two_by_two["pose only (RRR)"] = fit_and_score_pose_bottleneck(F_lag[tr], pose[tr], F_lag[te], pose[te], rrr_mean, rrr_comps, alpha_lag)
two_by_two["both (RRR)"] = fit_and_score_pose_bottleneck(____[tr], pose[tr], ____[te], pose[te], rrr_mean, rrr_comps, alpha_rrr)
two_by_two["random orth sEMG"] = fit_and_score_ridge(F_orth[tr], pose[tr], F_orth[te], pose[te], alpha_orth)
ceilings = {"eigenpostures": projection_ceiling(pose[te], pose_mean, pose_comps),
            "RRR pose": projection_ceiling(pose[te], rrr_mean, rrr_comps),
            "random": projection_ceiling(pose[te], pose_mean, ____)}
```

</details>

In [ ]:
# ✅ Check Task 2
r_Q, r_xm = ref.rrr_channel_basis(X_rrr[tr], pose[tr], K)
assert Q_rrr.shape == (16, K) and np.allclose(Q_rrr.T @ Q_rrr, np.eye(K), atol=1e-8), "Q should have orthonormal columns"
assert np.allclose(np.abs(Q_rrr.T @ r_Q), np.eye(K), atol=1e-6) or np.allclose(Q_rrr @ Q_rrr.T, r_Q @ r_Q.T, atol=1e-6), \
    "the channel subspace differs from the reference"
r_m, r_c = ref.rrr_pose_basis(X_rrr[tr], pose[tr], K_POSE)
assert rrr_comps.shape == (K_POSE, 20) and np.allclose(rrr_comps.T @ rrr_comps, r_c.T @ r_c, atol=1e-6)
assert np.isclose(ceilings["eigenpostures"], ref.projection_ceiling(pose[te], pose_mean, pose_comps))
e = ref.fit_and_score_pose_bottleneck(F_rrr[tr], pose[tr], F_rrr[te], pose[te], rrr_mean, rrr_comps, alpha_rrr)
assert np.isclose(two_by_two["both (RRR)"]["angle_deg"], e["angle_deg"]), "'both (RRR)' differs from the reference"
for name, s in two_by_two.items():
    print(f"{name:22s} {s['angle_deg']:6.2f}°   R² = {s['r2']:6.3f}")
print("Pose-bottleneck ceilings (true test poses squeezed through K_POSE directions): "
      + ", ".join(f"{k} {v:.2f}°" for k, v in ceilings.items()))
print(f"✅ Angles between the RRR channel subspace and the synergy subspace: "
      f"{ref06.principal_angles(Q_rrr, ref06.orthonormal(W_syn.T)).round(0)}°")

### 📊 The full table: no bottleneck, variance-based, pose-informed, random
**What to look for:** (1) at the same size k, which sEMG bottleneck stays close to "neither"? (2) does each one beat
its random control? (3) the pose bottlenecks: how far is each from its ceiling, and which loses less?

In [ ]:
order = ["neither", "sEMG only", "sEMG only (RRR)", "random sEMG", "random orth sEMG",
         "pose only", "pose only (RRR)", "both", "both (RRR)", "random both"]
colors = ["C0", "C1", "C2", "0.6", "0.6", "C1", "C2", "C1", "C2", "0.6"]
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.bar(order, [two_by_two[n]["angle_deg"] for n in order], color=colors)
ax.axhline(results["global median"]["angle_deg"], ls="--", color="gray", label="global median")
ax.axhline(results["per-recording median"]["angle_deg"], ls=":", color="gray", label="per-recording median")
ax.set_ylim(min(two_by_two[n]["angle_deg"] for n in order) - 1, results["global median"]["angle_deg"] + 1)
ax.set_ylabel("test angular error (°)"); ax.tick_params(axis="x", rotation=30); ax.legend(fontsize=8)
ax.set_title("orange = by variance (NMF / eigenpostures) · green = for the pose (RRR) · grey = random")
plt.tight_layout(); plt.show()

✍️ **Q2.** At k = 4, which sEMG bottleneck keeps the decoder's information? The angles between the RRR subspace and the
synergy subspace are printed above: what do they tell you about *where* in the EMG the pose information sits?
RRR was allowed to look at the pose and NMF was not; is the comparison still useful, and for which question?

> *Your answer:*

---
## Task 3 · How small can each sEMG bottleneck be?
Repeat "sEMG only" for k = 2, 3, 4 and 8 with four kinds of bottleneck: NMF synergies, RRR directions, a random
non-negative basis and a random orthonormal basis. Use the same recipes as Tasks 1–2 (NMF on `emg[train][::5]`
with 200 rounds and `seed=SEED`; random bases from `np.random.default_rng(k)`), so k = 4 reproduces those numbers.

In [ ]:
K_SWEEP = [2, 3, 4, 8]
KINDS = ["NMF synergies", "RRR (for the pose)", "random non-negative", "random orthonormal"]
k_sweep: dict[str, list[float]] = {kind: [] for kind in KINDS}   # test angular error per k


def bottleneck_features(kind: str, k: int) -> np.ndarray:
    '''(N, k * 5) lagged features for one kind of sEMG bottleneck of size k, fit on the training frames.'''
    # TODO: one branch per kind (see the hints); every branch ends with lag_features(Z, rec, LAGS)
    raise NotImplementedError


# TODO: for each k in K_SWEEP and each kind: F = bottleneck_features(kind, k); choose alpha on tr;
#       append fit_and_score_ridge(...)["angle_deg"] to k_sweep[kind]

#### 💡 Stuck on Task 3? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `bottleneck_features` builds `Z` (N × k) four ways, then returns `lag_features(Z, rec, LAGS)`:
   - NMF: `nmf_fit` on `emg[train][::5]`, `nmf_transform` on all frames, `np.log(C + 1e-3)`.
   - RRR: `rrr_channel_basis(X_rrr[tr], pose[tr], k)`, then `(log_emg - x_mean) @ Q`.
   - random non-negative: `np.random.default_rng(k).random((k, 16))`, rows to length 1, `np.log(emg @ W.T + 1e-3)`.
   - random orthonormal: `ref06.random_subspace(16, k, np.random.default_rng(k))` on centred `log_emg` (as `F_orth`).
2. A double loop: `for k in K_SWEEP: for kind in KINDS:`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- Branch on a string with `if kind == "NMF synergies": … elif …`.

```python
for k in [2, 4]:
    for kind in ["a", "b"]:
        print(k, kind)
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def bottleneck_features(kind, k):
    if kind == "NMF synergies":
        _, W, _ = nmf_fit(emg[train][::5], k, n_iter=200, seed=SEED)
        Z = np.log(nmf_transform(emg, W, n_iter=200, seed=SEED) + 1e-3)
    elif kind == "RRR (for the pose)":
        Q, xm = rrr_channel_basis(X_rrr[tr], pose[tr], ____)
        Z = (log_emg - xm) @ Q
    elif kind == "random non-negative":
        W = np.random.default_rng(k).random((k, 16)); W /= np.linalg.norm(W, axis=1)[:, None]
        Z = np.log(emg @ W.T + 1e-3)
    else:
        Q = ref06.random_subspace(16, k, np.random.default_rng(____))
        Z = (log_emg - log_emg[train].mean(axis=0)) @ Q
    return lag_features(Z, rec, LAGS)

for k in K_SWEEP:
    for kind in KINDS:
        F = bottleneck_features(kind, k)
        a = choose_alpha(F[tr], pose[tr], rec[tr])
        k_sweep[kind].append(fit_and_score_ridge(F[tr], pose[tr], F[te], pose[te], a)["____"])
```

</details>

In [ ]:
# ✅ Check Task 3
assert all(len(v) == len(K_SWEEP) for v in k_sweep.values()), "one error per k for each kind"
i4 = K_SWEEP.index(4)
assert np.isclose(k_sweep["NMF synergies"][i4], two_by_two["sEMG only"]["angle_deg"]), "k = 4 should reproduce Task 1"
assert np.isclose(k_sweep["RRR (for the pose)"][i4], two_by_two["sEMG only (RRR)"]["angle_deg"]), "k = 4 should reproduce Task 2"
assert np.isclose(k_sweep["random orthonormal"][i4], two_by_two["random orth sEMG"]["angle_deg"])
for kind, errs in k_sweep.items():
    print(f"{kind:22s} " + "  ".join(f"k={k}: {e:5.2f}°" for k, e in zip(K_SWEEP, errs)))
print("✅ Size sweep done")

### 📊 Decoder error against bottleneck size
**What to look for:** the smallest k at which each curve reaches the dashed "no bottleneck" line, and whether the
learned curves stay below their random controls.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for kind, c, ls in zip(KINDS, ["C1", "C2", "0.5", "0.5"], ["-", "-", "--", ":"]):
    ax.plot(K_SWEEP, k_sweep[kind], marker="o", color=c, ls=ls, label=kind)
ax.axhline(two_by_two["neither"]["angle_deg"], ls="--", color="C0", lw=0.8, label="no bottleneck (16 ch)")
ax.set_xlabel("k (size of the sEMG bottleneck)"); ax.set_ylabel("test angular error (°)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

✍️ **Q3.** How many dimensions does each kind of bottleneck need to come within ~0.5° of no bottleneck? Compare with the k
that notebook 02 chose from VAF. Why can "the k that rebuilds the EMG" and "the k the decoder needs" be so different?

> *Your answer:*

---
## Task 4 · Calibration curves: does a bottleneck need less data on a new task?
H1 is about **new users**, which one person can't test. The closest within-person version is a **new task**: hold
out a stage, train on the others, give the decoder the first *s* seconds of the new stage, and score it on the rest.
`protocol.calibration_splits(test_mask, rec, seconds)` splits each held-out recording into a **calibration pool**
(first half) and an **evaluation set** (second half, the same for every budget).

Two ways to use the calibration frames:
- **Pooled:** add them to the training set and refit. A few seconds among ~30 minutes of other data barely move
  the weights.
- **Offset only:** keep the weights; shift every prediction by the mean error on the calibration frames. This refits
  just a handful of numbers (like Lazzari & Saxena refitting only an input vector).

Conditions: "neither", "both NMF" (synergies + eigenpostures), "both RRR" (shared subspace on both sides) and
"random both". For an honest test **every bottleneck is refit in each fold on the training stages only**: NMF, RRR
and the eigenpostures must not see the held-out stage. (An earlier version fit the eigenpostures on all training
frames, which let the held-out stage's poses shape the pose bottleneck.) Budgets: 0, 5, 10 and 20 s per recording.
This takes about a minute.

In [ ]:
def offset_recalibrate(pred: np.ndarray, target: np.ndarray, cal: np.ndarray) -> np.ndarray:
    """Shift every prediction by the mean error on the calibration frames.

    Parameters
    ----------
    pred   : (N, J) float, predictions for all frames
    target : (N, J) float, true values
    cal    : (N,) bool, calibration frames

    Returns
    -------
    (N, J) float: pred + mean over cal frames of (target - pred). Unchanged if cal has no True entries.
    """
    # TODO
    raise NotImplementedError


K_CAL = K                          # the same small bottleneck as Tasks 1–2
BUDGETS = [0, 5, 10, 20]           # seconds of the held-out stage, per recording
CONDS = ["neither", "both NMF", "both RRR", "random both"]
cal_err = {(c, mode): {s: [] for s in BUDGETS} for c in CONDS for mode in ["pooled", "offset"]}

for name, trm, tem in protocol.leave_one_stage_out(stage_of_rec, rec, pose_valid, filename=filename):
    if tem.sum() == 0:
        continue
    fit = (~np.isin(rec, np.unique(rec[tem]))) & train                 # frames of the training stages only
    _, W_f, _ = nmf_fit(emg[fit][::10], K_CAL, n_iter=150, seed=SEED)
    Q_f, xm_f = rrr_channel_basis(X_rrr[fit & pose_valid], pose[fit & pose_valid], K_CAL)
    rm_f, rc_f = rrr_pose_basis(X_rrr[fit & pose_valid], pose[fit & pose_valid], K_POSE)
    pm_f, pc_f = pca_fit(pose[fit & pose_valid], K_POSE)
    W_r = np.random.default_rng(K_CAL).random((K_CAL, 16)); W_r /= np.linalg.norm(W_r, axis=1)[:, None]
    setups = {   # condition → (features for all frames, pose mean, pose directions or None)
        "neither":     (F_lag, None, None),
        "both NMF":    (lag_features(np.log(nmf_transform(emg, W_f, n_iter=150, seed=SEED) + 1e-3), rec, LAGS), pm_f, pc_f),
        "both RRR":    (lag_features((log_emg - xm_f) @ Q_f, rec, LAGS), rm_f, rc_f),
        "random both": (lag_features(np.log(emg @ W_r.T + 1e-3), rec, LAGS), pm_f, rand_comps),
    }
    for s, cal, ev in protocol.calibration_splits(tem, rec, BUDGETS):
        for c, (F, m_c, comps) in setups.items():
            # TODO: targets = pose, or the pose scores (pca_transform(pose, m_c, comps)) if comps is not None
            #       w, b = ridge_fit(F[trm | cal], targets[trm | cal], alpha_lag); pred = ridge_predict(F, w, b)
            #       pooled: back to angles (pca_inverse if comps is not None), angular_error on ev
            #       offset: offset_recalibrate(pred, targets, cal) first, then the same
            #       append both errors to cal_err[(c, "pooled")][s] and cal_err[(c, "offset")][s]
            pass

#### 💡 Stuck on Task 4? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `offset_recalibrate`: if `cal` has no True entries, return `pred`; otherwise add
   `(target[cal] - pred[cal]).mean(axis=0)`.
2. In the inner loop the targets are angles for "neither" and K_POSE scores for the others; fit on `trm | cal`,
   predict all frames, then score on `ev` twice: as is (pooled) and after `offset_recalibrate` (offset). Turn
   scores back into angles with `pca_inverse` before scoring.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- A tiny helper avoids repeating "back to angles":
  `back = (lambda Z: Z) if comps is None else (lambda Z: pca_inverse(Z, m_c, comps))`.

```python
pred = np.array([[1.0], [2.0], [3.0]]); target = np.array([[2.0], [3.0], [9.0]]); cal = np.array([True, True, False])
pred + (target[cal] - pred[cal]).mean(axis=0)     # [[2.], [3.], [4.]]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def offset_recalibrate(pred, target, cal):
    if not cal.any():
        return ____
    return pred + (target[____] - pred[cal]).mean(axis=0)

# the innermost loop body:
targets = pose if comps is None else pca_transform(pose, m_c, comps)
back = (lambda Z: Z) if comps is None else (lambda Z: pca_inverse(Z, m_c, comps))
m = trm | ____
w, b = ridge_fit(F[m], targets[m], alpha_lag)
pred = ridge_predict(F, w, b)
cal_err[(c, "pooled")][s].append(angular_error(pose[ev], back(pred)[ev]))
cal_err[(c, "offset")][s].append(angular_error(pose[ev], back(offset_recalibrate(pred, targets, ____))[ev]))
```

</details>

In [ ]:
# ✅ Check Task 4
p_t = np.array([[1.0], [2.0], [3.0]]); t_t = np.array([[2.0], [3.0], [9.0]]); c_t = np.array([True, True, False])
assert np.allclose(offset_recalibrate(p_t, t_t, c_t), ref.offset_recalibrate(p_t, t_t, c_t))
assert np.allclose(offset_recalibrate(p_t, t_t, np.zeros(3, bool)), p_t), "no calibration frames → unchanged"
n_folds = sum(te_.sum() > 0 for _, _, te_ in protocol.leave_one_stage_out(stage_of_rec, rec, pose_valid, filename=filename))
assert all(len(v) == n_folds for d_ in cal_err.values() for v in d_.values()), "one error per fold and budget"
assert np.allclose(cal_err[("neither", "pooled")][0], cal_err[("neither", "offset")][0]), \
    "with a 0 s budget the two modes must agree (nothing to recalibrate)"
for key, d_ in cal_err.items():
    print(f"{key[0]:12s} {key[1]:7s} " + "  ".join(f"{s:2d}s: {np.mean(v):5.2f}°" for s, v in d_.items()))
diff = np.array(cal_err[("both RRR", "offset")][5]) - np.array(cal_err[("neither", "offset")][5])
m, lo, hi = protocol.session_bootstrap_ci(diff)
print(f"✅ Calibration curves ready. both RRR − neither at 5 s (offset): {m:+.2f}° (95% CI {lo:+.2f} to {hi:+.2f}, over folds)")

### 📊 Calibration curves
Mean error on the evaluation half of the held-out stages against seconds of calibration per recording, with 95%
CIs over folds. **What to look for:** does any curve fall quickly? Is a bottleneck curve **below** "neither" at small
budgets (it needs less data), or only parallel to it?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for i, mode in enumerate(["pooled", "offset"]):
    for j, c in enumerate(CONDS):
        stats = [protocol.session_bootstrap_ci(np.array(cal_err[(c, mode)][s])) for s in BUDGETS]
        ax[i].plot(BUDGETS, [m for m, _, _ in stats], marker="o", color=f"C{j}", label=c)
        ax[i].fill_between(BUDGETS, [lo for _, lo, _ in stats], [hi for _, _, hi in stats], color=f"C{j}", alpha=0.15)
    ax[i].set_title(f"{mode} calibration"); ax[i].set_xlabel("calibration seconds per recording")
ax[0].set_ylabel("angular error on the held-out half (°)"); ax[0].legend(); plt.tight_layout(); plt.show()

✍️ **Q4.** Does any bottleneck make calibration on a new task **faster** (a curve below "neither" at 5–10 s), or only cheaper
(the same curve with fewer dimensions)? Why might offset-only calibration get *worse* at 5 s before it gets better?
What would you change before running this on held-out **users** (stage 7)?

> *Your answer:*

---
## Result · gate for stage 6
**Question:** Do the bottlenecks keep the decoder's information, does it matter how they are chosen, and do they reduce the data needed on a new task?

| Measure | Value (mean, 95% CI over sessions) | Baseline / null | Pass? |
|---|---|---|---|
| neither / sEMG only (NMF) / sEMG only (RRR) at k = 4 (°) | | | |
| random non-negative / random orthonormal at k = 4 (°) | | | |
| pose only: eigenpostures / RRR pose; their ceilings (°) | | | |
| both (NMF + eigenpostures) / both (RRR) / random both (°) | | | |
| smallest k within 0.5° of neither: NMF vs RRR | | | |
| Calibration, offset, 5 s: both RRR − neither; both NMF − random both | | | |

**Gate:** a bottleneck of a few dimensions stays within ~0.5° of "neither", clearly beats its random control, and its calibration curve is at or below "neither" at small budgets. Record separately *which* bottleneck passes.

- **Pass →** worth testing on held-out **users** (full dataset, stage 7), with the pose-informed bottleneck (or a supervised NMF that keeps non-negative synergies but is trained to predict the pose, from notebook 06's extras) and the proposal's deep decoder.
- **Fail →** report it: on this person the bottlenecks cost accuracy without saving data. Before stage 7, try the stage-5 dynamical decoder (notebook 07), a supervised NMF, or a hierarchical (finger → hand) pose bottleneck.
- **Was your prediction right? Why or why not?**

---
## ⚠️ Methodology: what this notebook can and can't tell you
Read this before writing the Result. Each row is a reason the numbers could look better (or different) than the truth, and what would test it.

| Issue | Why it matters | What would test it |
|---|---|---|
| **One person, 15 blocks** | Every CI is over 15 blocks / 14 stage folds; the two hands of a block are simultaneous, so they are one unit | Repeat on several users of the full dataset (stage 7) |
| **RRR sees the pose; NMF doesn't** | The pose-informed bottleneck is supervised, so it *should* keep more pose information. The comparison shows where the information sits, not which synergies are 'real' | A supervised NMF (non-negative, trained with a pose term) compared at the same k |
| **Time split only for Tasks 1–3** | Bottlenecks are fit and scored on the same recordings (later in time); a new task can behave differently | Task 4 refits everything per held-out stage; extend Tasks 1–3 to leave-one-stage-out |
| **Linear lagged decoder** | A bottleneck can only lose what the decoder uses; a dynamical or nonlinear decoder might use other EMG directions | Rerun with the stage-5 decoder (notebook 07) |
| **Different transforms** | NMF and random non-negative latents are logged after projection; RRR and random orthonormal project the log-EMG | Both random controls are reported, one for each recipe |
| **Random controls are single draws** | One random basis can be lucky or unlucky | Average over several random seeds |
| **Alpha chosen once** | alpha_lag is reused in the calibration folds for every condition | Nested alpha selection inside each fold |
| **Offset calibration uses the first seconds** | The start of a task may not represent the rest, so small budgets can hurt | Draw calibration frames from across the pool |